# Variable Elimination

## 1. Objective

Variable Elimination is an exact inference algorithm for computing the posterior distribution of query variables given observed evidence.

It improves upon enumeration by storing intermediate factors and eliminating hidden variables one at a time, thereby avoiding repeated computations.

## 2. Mathematical Background

Let

- $Q$ be the query variables,
- $E=e$ be the observed evidence,
- $H$ be the hidden variables,
- $\phi_1,\ldots,\phi_k$ be the factors of the probabilistic model.

The posterior distribution is

$$
P(Q\mid e)
=
\alpha
\sum_H
\prod_{i=1}^{k}\phi_i,
$$

where $\alpha$ is a normalization constant.

Enumeration constructs the complete product before summing over all hidden-variable assignments. Variable Elimination changes the order of these operations.

For a hidden variable $Z$, collect all factors containing $Z$:

$$
\Phi_Z
=
\{\phi_i: Z\in\operatorname{scope}(\phi_i)\}.
$$

Multiply these factors:

$$
\psi
=
\prod_{\phi\in\Phi_Z}\phi.
$$

Then eliminate $Z$ through marginalization:

$$
\tau
=
\sum_Z \psi.
$$

The original factors in $\Phi_Z$ are replaced by the new factor $\tau$.

This process is repeated for every hidden variable. The remaining factors are multiplied and normalized to obtain the posterior distribution.

## 3. Pseudocode

```text
VARIABLE-ELIMINATION(query, evidence, factors, elimination_order)

    restrict every factor using the observed evidence

    for variable in elimination_order

        relevant_factors ← all factors containing variable

        remove relevant_factors from the factor collection

        product_factor ← multiply relevant_factors

        reduced_factor ← sum variable out of product_factor

        add reduced_factor to the factor collection

    result ← multiply all remaining factors

    remove any remaining non-query variables

    normalize result

    return result
```

## 4. Implementation

In [1]:
from collections.abc import Mapping, Sequence
from functools import reduce
from numbers import Real
from typing import Hashable

In [3]:
#%%capture
#%run "../../01_Representation/05_DiscreteFactor.ipynb"
#%run "../../01_Representation/06_Bayesian_Networks.ipynb"

### Helper: multiply a collection of factors

Variable Elimination repeatedly multiplies all factors containing the variable currently being eliminated.

In [3]:
def multiply_factors(
    factors: Sequence[Factor],
) -> Factor:
    """
    Multiply a non-empty sequence of factors.
    """
    if not factors:
        raise ValueError(
            "At least one factor is required."
        )

    return reduce(
        lambda left, right: left.multiply(right),
        factors,
    )

### Helper: restrict factors using evidence

For each observation $E_i=e_i$, every factor containing $E_i$ is restricted to the observed value.

After restriction, the evidence variable should no longer remain in that factor's scope.

In [4]:
def reduce_factors(
    factors: Sequence[Factor],
    evidence: Mapping[str, Hashable],
) -> tuple[list[Factor], float]:
    """
    Reduce factors using observed evidence.

    Scalar results are accumulated separately.

    Returns
    -------
    tuple[list[Factor], float]
        Remaining factors and the product of scalar constants.
    """
    reduced_factors: list[Factor] = []
    scalar_constant = 1.0

    for factor in factors:
        relevant_evidence = {
            variable: value
            for variable, value in evidence.items()
            if variable in factor.variables
        }

        if not relevant_evidence:
            reduced_factors.append(factor.copy())
            continue

        reduced = factor.reduce(relevant_evidence)

        if isinstance(reduced, Factor):
            reduced_factors.append(reduced)
        else:
            scalar_constant *= float(reduced)

    return reduced_factors, scalar_constant

### Variable Elimination

For each hidden variable:

1. Select all factors containing that variable.
2. Multiply the selected factors.
3. Sum out the variable.
4. Replace the selected factors with the resulting factor.

In [5]:
def variable_elimination(
    query_variables: Sequence[str],
    evidence: Mapping[str, Hashable],
    factors: Sequence[Factor],
    elimination_order: Sequence[str],
) -> Factor:
    """
    Compute an exact posterior distribution using Variable Elimination.

    Parameters
    ----------
    query_variables
        Variables that must remain in the posterior.
    evidence
        Observed variable-value assignments.
    factors
        Factors defining the probabilistic model.
    elimination_order
        Order in which hidden variables are eliminated.

    Returns
    -------
    Factor
        Normalized posterior factor over the query variables.
    """
    if not query_variables:
        raise ValueError(
            "At least one query variable is required."
        )

    query_set = set(query_variables)
    evidence_set = set(evidence)
    elimination_set = set(elimination_order)

    overlap = query_set & evidence_set

    if overlap:
        raise ValueError(
            "Query variables cannot also be evidence "
            f"variables: {overlap}."
        )

    invalid_eliminations = query_set & elimination_set

    if invalid_eliminations:
        raise ValueError(
            "Query variables cannot be eliminated: "
            f"{invalid_eliminations}."
        )

    working_factors, scalar_constant = reduce_factors(
        factors,
        evidence,
    )

    for variable in elimination_order:
        relevant_factors = [
            factor
            for factor in working_factors
            if variable in factor.variables
        ]

        if not relevant_factors:
            continue

        working_factors = [
            factor
            for factor in working_factors
            if variable not in factor.variables
        ]

        product_factor = multiply_factors(
            relevant_factors
        )

        reduced = product_factor.marginalize(variable)

        if isinstance(reduced, Factor):
            working_factors.append(reduced)
        else:
            scalar_constant *= float(reduced)

    if not working_factors:
        raise ValueError(
            "Inference produced only a scalar. "
            "Check that the query variables were not eliminated."
        )

    result = multiply_factors(working_factors)

    remaining_non_query = [
        variable
        for variable in result.variables
        if variable not in query_set
    ]

    for variable in remaining_non_query:
        reduced = result.marginalize(variable)

        if not isinstance(reduced, Factor):
            raise ValueError(
                "All variables were eliminated. "
                "Check the query variables."
            )

        result = reduced

    # Multiplying every value by the same positive scalar has no
    # effect after normalization, so this is mathematically optional.
    if not np.isclose(scalar_constant, 1.0):
        result = Factor(
            variables=result.variables.copy(),
            domains={
                variable: result.domains[variable].copy()
                for variable in result.variables
            },
            values=result.values * scalar_constant,
            name=f"{scalar_constant:g} × {result.name}",
        )

    return result.normalize()

## 5. Example

Consider the following Bayesian Network:

```text
Cloudy
  |
  v
Rain
  |
  v
WetGrass
```

The joint distribution factorizes as

$$
P(C,R,W)
=
P(C)\,P(R\mid C)\,P(W\mid R).
$$

Suppose we observe

$$
W=\text{True}
$$

and want to compute

$$
P(C\mid W=\text{True}).
$$

The variables are:

- Query: $C$
- Evidence: $W=\text{True}$
- Hidden: $R$

The elimination order therefore contains only $R$.

In [6]:
import numpy as np

In [7]:
cloudy_factor = Factor(
    variables=("Cloudy",),
    domains={
        "Cloudy": (True, False),
    },
    values=np.array([
        0.5,  # P(Cloudy=True)
        0.5,  # P(Cloudy=False)
    ]),
)

In [8]:
rain_factor = Factor(
    variables=("Cloudy", "Rain"),
    domains={
        "Cloudy": (True, False),
        "Rain": (True, False),
    },
    values=np.array([
        [0.8, 0.2],  # Cloudy=True
        [0.2, 0.8],  # Cloudy=False
    ]),
)

In [9]:
wet_grass_factor = Factor(
    variables=("Rain", "WetGrass"),
    domains={
        "Rain": (True, False),
        "WetGrass": (True, False),
    },
    values=np.array([
        [0.9, 0.1],  # Rain=True
        [0.1, 0.9],  # Rain=False
    ]),
)

In [10]:
posterior = variable_elimination(
    query_variables=["Cloudy"],
    evidence={
        "WetGrass": True,
    },
    factors=[
        cloudy_factor,
        rain_factor,
        wet_grass_factor,
    ],
    elimination_order=["Rain"],
)

posterior.print_table()

Cloudy | normalized((φ × Σ_Rain((φ × φ | {'WetGrass': True}))))
-------+-------------------------------------------------------
True   | 0.7400000000000001                                    
False  | 0.26                                                  


### Manual Verification

After applying the evidence $W=\text{True}$, the relevant factors are

$$
P(C),\qquad P(R\mid C),\qquad P(W=\text{True}\mid R).
$$

Eliminating $R$ gives

$$
\tau(C)
=
\sum_R
P(R\mid C)
P(W=\text{True}\mid R).
$$

For $C=\text{True}$:

$$
\tau(C=\text{True})
=
(0.8)(0.9)+(0.2)(0.1)
=
0.74.
$$

Multiplying by $P(C=\text{True})$:

$$
P(C=\text{True},W=\text{True})
=
(0.5)(0.74)
=
0.37.
$$

For $C=\text{False}$:

$$
\tau(C=\text{False})
=
(0.2)(0.9)+(0.8)(0.1)
=
0.26.
$$

Multiplying by $P(C=\text{False})$:

$$
P(C=\text{False},W=\text{True})
=
(0.5)(0.26)
=
0.13.
$$

The normalization constant is

$$
P(W=\text{True})
=
0.37+0.13
=
0.50.
$$

Therefore,

$$
P(C=\text{True}\mid W=\text{True})
=
\frac{0.37}{0.50}
=
0.74,
$$

and

$$
P(C=\text{False}\mid W=\text{True})
=
\frac{0.13}{0.50}
=
0.26.
$$

Hence,

$$
P(C\mid W=\text{True})
=
[0.74,\ 0.26].
$$

In [11]:
expected = np.array([0.74, 0.26])

assert np.allclose(
    posterior.values,
    expected,
), f"Expected {expected}, received {posterior.values}"

print("Variable Elimination test passed.")

Variable Elimination test passed.


## 6. Complexity Analysis

The complexity of Variable Elimination depends on the size of the largest intermediate factor created during elimination.

Let

- $d$ be the maximum variable-domain size,
- $w$ be the induced width associated with the elimination order.

Then the approximate time and space complexities are:

| Complexity | Value |
|---|---:|
| Time | $O(nd^{w+1})$ |
| Space | $O(d^{w+1})$ |

Variable Elimination can be substantially more efficient than enumeration, but its performance depends strongly on the graph structure and elimination order.

## 7. Implementation Notes

- Apply evidence restriction before eliminating hidden variables.
- Never eliminate a query variable.
- Multiply every current factor containing the variable being eliminated.
- Remove the multiplied factors before inserting the new marginalized factor.
- Add the marginalized factor back to the working factor collection.
- Normalize only after all required variables have been eliminated.
- Validate that evidence values belong to their corresponding domains.

## 8. Limitations

- The algorithm is exact but can still require exponential computation.
- Poor elimination orders can create very large intermediate factors.
- Finding the optimal elimination order is itself computationally difficult.
- Dense Bayesian Networks may make exact inference impractical.
- The method assumes that the complete probabilistic model and its factors are available.

## 9. Key Takeaways

- Variable Elimination computes exact posterior distributions.
- It avoids the repeated calculations performed by enumeration.
- Hidden variables are eliminated through factor multiplication followed by marginalization.
- Evidence is incorporated by restricting the original factors.
- The elimination order determines the size of intermediate factors.
- The final factor must be normalized to obtain a probability distribution.